# Interactive graph to show transmission effects in count rate on Eiger detectors

**CS1 – Precision at High Flux.** Move the transmission slider and switch crystal type.

* **Left – count rate.** Strong crystals at 100 % push the brightest pixels past the Dectris trusted limit, where the Eiger's count-rate correction can no longer recover the lost counts.
* **Right – dose.** Lowering transmission and lengthening exposure delivers **the same total dose** (here 10 MGy): same juice, just slower.

Detector curve: paralyzable dead-time model *m = n·exp(−n·τ)* fitted to DECTRIS' published EIGER count-rate data at 12.4 keV (figure in the CS1 case-study document). Dose: hard-coded from RADDOSE on I04 (10 MGy at 100 % transmission takes 16 s at 13 keV with the default beam, ~0.62 MGy/s), so no RADDOSE run is needed live.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown, Layout

In [ ]:
# ---------------- Detector (count rate) ----------------
# Beamline absolute max flux into the brightest pixels (arbitrary constant for demonstration)
MAX_BEAMLINE_FLUX = 1000.0
# Paralyzable dead time fitted to DECTRIS EIGER data at 12.4 keV (Mcps/mm2 axes): m = n * exp(-n * TAU)
TAU = 8.0e-4
# Dectris trusted limit expressed as a measured (uncorrected) rate
DECTRIS_LIMIT = 200.0

# ---------------- Dose (from GDA's RADDOSE on I04, hard-coded) ----------------
ENERGY_KEV = 13.0
TARGET_DOSE_MGY = 10.0
# Total exposure RADDOSE gives to reach TARGET_DOSE_MGY at 100 % transmission on I04
# (28 Sep 2026: 13 keV / 0.9537 A, 1.09e12 ph/s, 30.1 x 18.7 um beam -> 16.04 s for 10 MGy, ~0.62 MGy/s)
EXPOSURE_100_S = 16.0

# Dose rate scales linearly with transmission, so total exposure = EXPOSURE_100_S / T


def eiger_measured(true_rate):
    return true_rate * np.exp(-true_rate * TAU)


def true_rate_at_limit():
    # Invert m = n*exp(-n*tau) on the rising branch by bisection
    lo, hi = 0.0, 1.0 / TAU
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if eiger_measured(mid) < DECTRIS_LIMIT else (lo, mid)
    return lo


TRUE_RATE_AT_LIMIT = true_rate_at_limit()


def plot_detector_response(transmission, crystal_efficiency):
    t_frac = transmission / 100.0

    # 1. Flux on the crystal after the attenuators
    incident_on_crystal = MAX_BEAMLINE_FLUX * t_frac
    # 2. Flux diffracted by the crystal into the brightest pixels
    arriving_at_detector = incident_on_crystal * crystal_efficiency
    # 3. The rate the Eiger actually records (before its internal correction)
    measured_rate = eiger_measured(arriving_at_detector)
    recorded_pct = 100 * measured_rate / arriving_at_detector if arriving_at_detector else 100.0
    saturated = measured_rate > DECTRIS_LIMIT

    # Dose: exposure scales as 1/transmission so the total dose never changes
    total_time = EXPOSURE_100_S / t_frac
    total_time_100 = EXPOSURE_100_S

    fig, (ax, axd) = plt.subplots(1, 2, figsize=(15, 5.8), gridspec_kw={'width_ratios': [1.35, 1]})

    # ---- Left: count-rate linearity ----
    x = np.linspace(0, 600, 500)
    ax.plot(x, x, '--', color='gray', label='Theoretical Diffracted Flux (No Loss)', alpha=0.7)
    ax.plot(x, eiger_measured(x), '-', color='#0055ff', label='Measured Eiger Rate', linewidth=2.5)
    ax.axhline(y=DECTRIS_LIMIT, color='red', linestyle='--', linewidth=1.5, label='Dectris Trusted Limit')
    if saturated:
        ax.vlines(arriving_at_detector, measured_rate, arriving_at_detector,
                  color='red', alpha=0.4, linewidth=6, label='Counts lost')
    ax.plot(arriving_at_detector, measured_rate, 'ro' if saturated else 'go', markersize=10, label='Current State')
    ax.set_title('Eiger Count Rate Linearity', fontsize=14)
    ax.set_xlabel('Diffracted Flux Arriving at Detector (Mcps/mm²)', fontsize=12)
    ax.set_ylabel('Measured Count Rate (Mcps/mm²)', fontsize=12)
    ax.set_xlim(0, 600)
    ax.set_ylim(0, 450)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=9)
    if saturated:
        ax.text(330, 410, 'WARNING: Data Quality Compromised!', color='red', fontsize=11,
                bbox=dict(facecolor='white', edgecolor='red', alpha=0.9))
    else:
        ax.text(330, 410, 'STATUS: Accurate Photon Counting', color='green', fontsize=11,
                bbox=dict(facecolor='white', edgecolor='green', alpha=0.9))

    # ---- Right: accumulated dose vs time ----
    t_max = max(total_time, total_time_100) * 1.1
    axd.plot([0, total_time_100], [0, TARGET_DOSE_MGY], '-', color='gray', lw=2, alpha=0.7,
             label=f'100 % transmission: {total_time_100:.1f} s')
    axd.plot([0, total_time], [0, TARGET_DOSE_MGY], '-', color='red' if saturated else 'green', lw=3,
             label=f'{transmission:.0f} % transmission: {total_time:.1f} s')
    axd.axhline(TARGET_DOSE_MGY, color='black', ls=':', lw=1.5)
    axd.text(t_max * 0.02, TARGET_DOSE_MGY * 1.03, f'Same total dose: {TARGET_DOSE_MGY:.0f} MGy', fontsize=11)
    axd.set_xlim(0, t_max)
    axd.set_ylim(0, TARGET_DOSE_MGY * 1.25)
    axd.set_title('Same Dose, Just Slower', fontsize=14)
    axd.set_xlabel('Collection time (s)', fontsize=12)
    axd.set_ylabel('Accumulated dose (MGy)', fontsize=12)
    axd.grid(True, alpha=0.3)
    axd.legend(loc='lower right', fontsize=10)

    plt.tight_layout()
    plt.show()

    # Highest whole-percent transmission that keeps this crystal under the limit
    safe_transmission = float(min(100, np.floor(100.0 * TRUE_RATE_AT_LIMIT / (MAX_BEAMLINE_FLUX * crystal_efficiency))))

    print("--- Count Rate ---")
    print(f"Transmission:           {transmission:.0f}%")
    print(f"Incident on Crystal:    {incident_on_crystal:.1f} Mcps/mm²")
    print(f"Arriving at Detector:   {arriving_at_detector:.1f} Mcps/mm² (Theoretical)")
    print(f"Recorded by Eiger:      {measured_rate:.1f} Mcps/mm² (Measured, {recorded_pct:.0f}% of arriving)")
    if saturated:
        print("Status:                 SATURATED (Lower transmission required!)")
        print(f"Safe transmission:      <= {safe_transmission:.0f}%")
    else:
        print("Status:                 SAFE (Eiger correction reliable)")
    print()
    print(f"--- Dose ({ENERGY_KEV:.0f} keV, I04 default beam, RADDOSE) ---")
    print(f"Total exposure:         {total_time:.1f} s   (x{100 / transmission:.1f} vs 100%)")
    print(f"Total dose:             {TARGET_DOSE_MGY:.1f} MGy  (unchanged)")


# Relative diffracting power of the crystal's strongest spots (illustrative scale, not a real % of photons)
crystal_options = [
    ('Small / Weakly Diffracting', 0.05),
    ('Medium / Average Crystal', 0.20),
    ('Massive / Highly Diffracting', 0.60)
]

_ = interact(plot_detector_response,
             transmission=FloatSlider(value=100, min=1, max=100, step=1,
                                      description='Transmission %:',
                                      style={'description_width': 'initial'},
                                      layout=Layout(width='600px')),
             crystal_efficiency=Dropdown(options=crystal_options,
                                         value=0.60,
                                         description='Crystal Type:',
                                         style={'description_width': 'initial'},
                                         layout=Layout(width='600px')))